# 02 — Clean with `drop_reason`

Auditable cleaning: every dropped row gets a single explicit reason, and the function returns both the cleaned data and a summary of what was thrown away.

In [ ]:
import pandas as pd

from geo_daily_tools.sample_data import messy_sensor_records
from geo_daily_tools.validation import (
    strip_string_columns,
    valid_lat_lon_mask,
    validate_sensor_records,
    flag_outliers_iqr,
    range_check,
    value_in_set_check,
)

df = pd.DataFrame(messy_sensor_records())
df

## Strip whitespace and convert blanks to NA

In [ ]:
strip_string_columns(df, ['obs_id'])[['obs_id']]

## Filter to rows with valid lat/lon

In [ ]:
df.loc[valid_lat_lon_mask(df)]

## End-to-end QA: `validate_sensor_records`

Returns valid rows, invalid rows (with `drop_reason`), and a summary dict.

In [ ]:
valid_df, invalid_df, summary = validate_sensor_records(
    messy_sensor_records(), return_invalid=True
)
summary

In [ ]:
valid_df

In [ ]:
invalid_df[['obs_id', 'lat', 'lon', 'reading', 'drop_reason']]

## Range and allowed-value checks

In [ ]:
range_check(
    valid_df,
    {'lat': (-90, 90), 'lon': (-180, 180), 'reading': (0, 100)},
)

In [ ]:
value_in_set_check(valid_df, {'sensor_type': {'type_a', 'type_b', 'type_c'}})

## Flag IQR outliers, optionally per group

In [ ]:
flagged = valid_df.copy()
flagged['is_outlier'] = flag_outliers_iqr(flagged, 'reading', group_cols='sensor_type')
flagged[['obs_id', 'sensor_type', 'reading', 'is_outlier']]